# L08 – Lösningar
Kräver `dlkit/train.py` i L08-versionen (med `scheduler`).

In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from dlkit.seed import set_seed
from dlkit.data import fashion_mnist_loaders
from dlkit.train import fit, evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
loss_fn = nn.CrossEntropyLoss()

def make_mlp(hidden=256, dropout=0.0):
    return nn.Sequential(nn.Flatten(),
                         nn.Linear(784, hidden), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(hidden // 2, 10))

### Övning 1

In [ ]:
def sample_log_uniform(low, high, rng):
    return 10 ** rng.uniform(np.log10(low), np.log10(high))

r = np.random.default_rng(0)
samples = np.array([sample_log_uniform(1e-5, 1e-1, r) for _ in range(20_000)])
assert samples.min() >= 1e-5 and samples.max() <= 1e-1
assert abs(np.median(samples) - 1e-3) / 1e-3 < 0.1
assert abs((samples < 1e-4).mean() - 0.25) < 0.02
print("Rätt!")

### Övning 2

In [ ]:
rng = np.random.default_rng(1)

def sample_config(rng):
    return {
        "lr": sample_log_uniform(1e-4, 1e-2, rng),
        "weight_decay": sample_log_uniform(1e-4, 1e-1, rng),
        "hidden": int(rng.choice([128, 256, 512])),
        "dropout": float(rng.uniform(0.0, 0.5)),
        "batch_size": int(rng.choice([32, 64, 128, 256])),
    }

trials = []
for trial in range(8):
    cfg = sample_config(rng)
    set_seed(42)
    tl, vl, _ = fashion_mnist_loaders(batch_size=cfg["batch_size"], train_subset=10_000)
    model = make_mlp(cfg["hidden"], cfg["dropout"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    h = fit(model, tl, vl, loss_fn, opt, device, epochs=8, patience=3, verbose=False)
    trials.append({**cfg, "val_loss": min(h["val_loss"]), "val_acc": max(h["val_acc"])})

df = pd.DataFrame(trials).sort_values("val_loss")
df.round(4)

**Tolkning:** inom intervallet 32–256 har batchstorleken oftast mindre betydelse än learning rate. Den samverkar dock med den: en liten batch med hög learning rate kan bli instabil, och en stor batch med låg learning rate lär sig långsamt. Med bara 8 försök är det svårt att dra säkra slutsatser. Det är ett skäl att fixera batchstorleken och söka på de andra parametrarna.

### Övning 3
Vi använder den bästa konfigurationen från övning 2. Med lektionens `df` fungerar det på samma sätt.

In [ ]:
best = df.iloc[0]
print(best)

train_loader, val_loader, test_loader = fashion_mnist_loaders(batch_size=int(best.get("batch_size", 128)))
set_seed(42)
model = make_mlp(int(best["hidden"]), float(best["dropout"])).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=float(best["lr"]), weight_decay=float(best["weight_decay"]))
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
h = fit(model, train_loader, val_loader, loss_fn, opt, device, epochs=30, patience=5, scheduler=sched)

final_test_acc = evaluate(model, test_loader, loss_fn, device)[1]   # en gång!
assert final_test_acc > 0.87, final_test_acc
print("Rätt! test accuracy:", round(final_test_acc, 3))